## Calling AI model with Groq with Structured response from LLM using Pydantic.

In [20]:
from pydantic_settings import BaseSettings, SettingsConfigDict
from pydantic import Field, SecretStr
from pydantic import BaseModel, ValidationError
import json 

class AppSettings(BaseSettings):
    model_config = SettingsConfigDict(env_file="../.env")
    groq_api_key: SecretStr

In [5]:
settings = AppSettings()   # reads from .env / environment automatically
print(settings.groq_api_key)      

**********


In [16]:
def call_groq(question: str) -> str:
    from openai import OpenAI

    client = OpenAI(api_key=settings.groq_api_key.get_secret_value(), base_url="https://api.groq.com/openai/v1")
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile", max_tokens=200, messages=[{"role": "user", "content": question}]
    )
    return response.choices[0].message.content

In [ ]:
class WeatherInfo(BaseModel):
    """The only shape we're willing to trust back from the model: a plain
    city name plus whether the person wants Fahrenheit instead of Celsius.
    """
    city: str 
    wants_fahrenheit: bool = False 


### Asks the model to respond with ONLY a JSON object matching
    WeatherInfo, then parses and validates it. Returns the validated
    model on success, or a short error string if the reply didn't match
    the expected shape -- so a bad response never silently continues.

In [ ]:
def extract_weather_question(user_message: str) -> WeatherInfo | str:
    instruction = (
        "Read the user's message and reply with ONLY a JSON object -- no other "
        'text -- in this exact shape: {"city": "<city name>", '
        '"wants_fahrenheit": <true or false>}. '
        f"User's message: {user_message!r}"
    )
    raw_reply = call_groq(instruction)

    try:
        cleaned = raw_reply.strip().removeprefix("```json").removesuffix("```").strip()
        data = json.loads(cleaned)
        return WeatherInfo(** data)
    except (json.JSONDecodeError, ValidationError) as exc:
        return f"Rejected: {exc}"

In [21]:
for message in [
    "What's the weather like in Tokyo right now?",
    "Is it warm in Delhi today? I want it in Fahrenheit please.",
]:
    result = extract_weather_question(message)
    print(f"{message!r} -> {result}")

 
 
 
 Raw reply from model: '{"city": "Tokyo", "wants_fahrenheit": false}'
"What's the weather like in Tokyo right now?" -> city='Tokyo' wants_fahrenheit=False
 
 
 
 Raw reply from model: '{"city": "Delhi", "wants_fahrenheit": true}'
'Is it warm in Delhi today? I want it in Fahrenheit please.' -> city='Delhi' wants_fahrenheit=True
